# Hybrid CV-DV LCHS: Reader Walkthrough

This notebook is a compact guide to the algorithm used in the manuscript. It shows the key objects that enter the hybrid CV-DV LCHS workflow without requiring the reader to inspect the full codebase.

The notebook has two roles:

1. run one small live example that finishes quickly,
2. load the retained paper summaries that contain the final Section 6 numbers.

The live example below is intentionally cheaper than the paper setting: it uses 10 Trotter steps for speed. The paper results loaded later in the notebook use 100 Trotter steps.


## Algorithm At A Glance

For a semidiscrete linear system
\[
\frac{d u(t)}{dt} = -A u(t), \qquad A = L + iH,
\]
the hybrid method uses a continuous-variable mode and a qubit register to realize the postselected map
\[
\langle \phi | e^{-it(\hat x \otimes L + I \otimes H)} | \psi \rangle.
\]

In the clean implementation, the workflow is:

1. build the semidiscrete heat operator for a chosen boundary condition,
2. choose kernel parameters \(r_{\mathrm{target}}, r', \beta, n_{\mathrm{coeff}}\),
3. compute the truncated oscillator coefficient state,
4. prepare that oscillator seed by `injection`, `givens`, or `SNAP+D`,
5. run the Trotterized hybrid evolution and postselect the oscillator on \(\ket{0}\),
6. compare the output against the exact PDE solution and the exact finite-Fock reference.

The main fidelities are
\[
F_{\mathrm{exact}} = F(u_{\mathrm{alg}}, u_{\mathrm{exact}}), \qquad
F_{\mathrm{trunc}} = F(u_{\mathrm{alg}}, u_{\mathrm{trunc}}), \qquad
F_{\mathrm{oracle}} = F(\psi_{\mathrm{prep}}, \psi_{\mathrm{target}}).
\]

Here:

- \(F_{\mathrm{oracle}}\) measures oscillator state-preparation quality,
- \(F_{\mathrm{exact}}\) measures end-to-end PDE quality,
- \(F_{\mathrm{trunc}}\) checks whether the hybrid evolution accurately implements the prepared finite-Fock model.


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import display

from clean_core import (
    KernelSpec,
    StatePrepSpec,
    EvolutionSpec,
    build_dirichlet_heat_system,
    build_neumann_heat_system,
    build_periodic_heat_system,
    compute_lchs_coefficients,
    generator_matrix,
    system_blocks,
)
from clean_oracles import prepare_cv_oracle
from clean_hybrid import run_clean_lchs


def show_rows(rows):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        from pprint import pprint
        pprint(rows)


def load_best_row(path):
    return json.loads(Path(path).read_text())["best_row"]


## 1. Build The Heat-Equation Benchmark

The paper studies the one-dimensional heat equation on a two-qubit register with Dirichlet, Neumann, and periodic boundary conditions. The next cells construct the three semidiscrete generators and show the Pauli-sum structure actually used by the hybrid simulator.


In [ ]:
systems = {
    "dirichlet": build_dirichlet_heat_system(
        num_qubits=2, alpha=1.0, grid_spacing=1.0, total_time=1.0, init_basis_index=1
    ),
    "neumann": build_neumann_heat_system(
        num_qubits=2, alpha=1.0, grid_spacing=1.0, total_time=1.0, init_basis_index=1
    ),
    "periodic": build_periodic_heat_system(
        num_qubits=2, alpha=1.0, grid_spacing=1.0, total_time=1.0, init_basis_index=1
    ),
}

summary_rows = []
for name, system in systems.items():
    A = np.real_if_close(generator_matrix(system))
    L, H = system_blocks(system)
    summary_rows.append(
        {
            "boundary_condition": name,
            "matrix_shape": A.shape,
            "nonzero_L_terms": sum(abs(term.coeff) > 1e-12 for term in system.l_terms),
            "nonzero_H_terms": sum(abs(term.coeff) > 1e-12 for term in system.h_terms),
            "spectral_norm_L": float(np.linalg.norm(L, 2)),
        }
    )
show_rows(summary_rows)


In [ ]:
for name, system in systems.items():
    print(f"\n{name.title()} generator A:")
    print(np.real_if_close(generator_matrix(system)))


In [ ]:
pauli_rows = []
for name, system in systems.items():
    for term in system.l_terms:
        if abs(term.coeff) > 1e-12:
            pauli_rows.append(
                {
                    "boundary_condition": name,
                    "block": "L",
                    "label": term.label,
                    "coeff": float(np.real_if_close(term.coeff)),
                }
            )
show_rows(pauli_rows)


## 2. Choose A Kernel And Inspect The Target Oscillator State

The CV oracle is specified by the kernel parameters \(r_{\mathrm{target}}, r', \beta, n_{\mathrm{coeff}}\). The coefficient state is
\[
\psi_{\mathrm{target}} \approx \sum_{n=0}^{n_{\mathrm{coeff}}-1} c_n \ket{n}.
\]

For the live example we use a small setting so that the notebook remains quick to run.


In [ ]:
demo_kernel = KernelSpec(
    r_target=2.0,
    r_prime=0.8,
    beta=0.8,
    n_coeff=8,
    n_fock=16,
    n_quad=24,
)

coeffs = compute_lchs_coefficients(demo_kernel)
coeff_rows = []
for n, coeff in enumerate(coeffs[: demo_kernel.n_coeff]):
    coeff_rows.append(
        {
            "n": n,
            "Re(c_n)": float(np.real(coeff)),
            "Im(c_n)": float(np.imag(coeff)),
            "|c_n|": float(abs(coeff)),
        }
    )

show_rows(coeff_rows)
print("Coefficient vector norm:", float(np.linalg.norm(coeffs)))


## 3. Compare The Three Oracle-Preparation Routes

The same target oscillator state can be prepared three different ways:

- `injection`: ideal direct loading,
- `givens`: analytic exact synthesis in the truncated oscillator space,
- `SNAP+D`: variational non-Gaussian preparation.

For the `SNAP+D` demonstration we use only two layers and a very small optimizer budget so that the example finishes quickly.


In [ ]:
np.random.seed(0)

prep_specs = {
    "injection": StatePrepSpec(method="injection"),
    "givens": StatePrepSpec(method="givens"),
    "snap_d_depth2": StatePrepSpec(method="snap_d", snap_depth=2, snap_restarts=1, snap_maxiter=25),
}

oracles = {}
oracle_rows = []
for name, spec in prep_specs.items():
    oracle = prepare_cv_oracle(demo_kernel, spec, coeffs=coeffs)
    oracles[name] = oracle
    meta = oracle.metadata
    oracle_rows.append(
        {
            "method": name,
            "apply_mode": oracle.apply_mode,
            "F_oracle": float(oracle.oracle_fidelity),
            "active_fock_levels": meta.get("n_active_fock_levels", ""),
            "JC_pulses": meta.get("n_jc_pulses", ""),
            "qubit_rotations": meta.get("n_qubit_rotations", ""),
            "snap_depth": meta.get("snap_depth", ""),
            "snap_total_iterations": meta.get("snap_total_iterations", ""),
        }
    )

show_rows(oracle_rows)


## 4. Run One End-To-End Hybrid Simulation

Now we fix the Dirichlet system and run the full hybrid algorithm on the same toy setting. The purpose is not to reproduce the paper-scale numbers, but to show how the three fidelities behave in one concrete example.

For speed, this tutorial example uses 10 Trotter steps rather than the 100-step setting used in the paper.


In [ ]:
demo_system = systems["dirichlet"]
demo_evolution = EvolutionSpec(n_trotter_steps=10)

run_rows = []
run_results = {}
for name, spec in prep_specs.items():
    result = run_clean_lchs(
        demo_system,
        demo_kernel,
        spec,
        demo_evolution,
        coeffs=coeffs,
        oracle=oracles[name],
    )
    run_results[name] = result
    run_rows.append(
        {
            "method": name,
            "F_oracle": float(result.oracle_fidelity),
            "F_exact": float(result.fidelity_vs_exact),
            "F_trunc": float(result.fidelity_vs_truncated),
            "postselection": float(result.postselection_probability),
            "circuit_depth": int(result.circuit_depth),
            "circuit_size": int(result.circuit_size),
        }
    )

show_rows(run_rows)


In [ ]:
snap_result = run_results["snap_d_depth2"]

print("Operation counts for the toy SNAP+D run:")
print(snap_result.count_ops)
print("\nOracle metadata:")
print(snap_result.metadata["oracle_metadata"])


The toy example already shows the qualitative interpretation used in the paper:

- when \(F_{\mathrm{trunc}}\) is close to one, the hybrid evolution is accurate for the prepared seed,
- the remaining gap between \(F_{\mathrm{oracle}}\) and \(F_{\mathrm{exact}}\) is then mainly a state-preparation effect.

This notebook example uses a reduced 10-step Trotterization for speed. The retained paper runs loaded below use the full 100-step setting.


## 5. Load The Retained Paper Results

The manuscript uses three retained datasets:

- `results_clean_refined_*` for ideal loading,
- `results_clean_givens_*` for the analytic Givens baseline,
- `results_clean_snap_recreate_*` for the final replayable `SNAP+D` runs.

The next cell loads the saved best row from each folder and reconstructs the main Section 6 comparison table.


In [ ]:
paper_rows = []
for bc in ["dirichlet", "neumann", "periodic"]:
    ideal = load_best_row(f"results_clean_refined_{bc}/sweep_summary.json")
    givens = load_best_row(f"results_clean_givens_{bc}/sweep_summary.json")
    snap = load_best_row(f"results_clean_snap_recreate_{bc}/sweep_summary.json")

    paper_rows.append(
        {
            "boundary_condition": bc,
            "r_target": ideal["r_target"],
            "r_prime": ideal["r_prime"],
            "beta": ideal["beta"],
            "n_coeff": ideal["n_coeff"],
            "n_trotter_steps": ideal["n_trotter_steps"],
            "F_exact (injection)": ideal["fidelity"],
            "F_givens": givens["fidelity"],
            "F_snap": snap["fidelity"],
            "F_oracle": snap["oracle_fidelity"],
            "F_trunc (snap)": snap["fidelity_vs_truncated"],
            "p_post (snap)": snap["postselection_probability"],
            "snap_depth": snap["snap_depth"],
        }
    )

show_rows(paper_rows)


These saved rows are the paper-scale message in compact form:

- ideal loading already reaches fidelity above 0.9989 in all three boundary conditions,
- analytic Givens synthesis matches that ideal baseline to numerical precision,
- optimized `SNAP+D` also reaches very high end-to-end fidelity in all three cases,
- the truncated-model fidelity remains essentially one, so the hybrid evolution is not the bottleneck once the oscillator seed is fixed.


## 6. Load The Retained DV Comparison Summaries

The manuscript also compares the hybrid approach against a retained DV LCHS reference for the Dirichlet benchmark. The saved JSON summaries already contain the circuit-level quantities used in the paper, so the notebook only needs to read them.


In [ ]:
dv_one_step = json.loads(Path("results_clean_dv_lchs_dirichlet/default_summary.json").read_text())
dv_trotter100 = json.loads(Path("results_clean_dv_lchs_dirichlet/trotter100_summary.json").read_text())

dv_rows = [
    {"quantity": "DV branch count", "value": dv_one_step["lcu_branch_count"]},
    {"quantity": "DV control qubits", "value": dv_one_step["num_qubits_lcu"]},
    {"quantity": "MPS prep depth", "value": dv_trotter100["kernel_prep_depth"]},
    {"quantity": "MPS prep size", "value": dv_trotter100["kernel_prep_size"]},
    {"quantity": "One-slice controlled-evolution depth", "value": dv_trotter100["controlled_evolution_depth"]},
    {"quantity": "One-slice controlled-evolution size", "value": dv_trotter100["controlled_evolution_size"]},
    {"quantity": "Full 100-slice circuit depth", "value": dv_trotter100["circuit_depth"]},
    {"quantity": "Full 100-slice circuit size", "value": dv_trotter100["circuit_size"]},
]

show_rows(dv_rows)


## 7. What To Take Away

This notebook is meant to separate two tasks clearly:

1. use the toy example to see the algorithmic objects directly,
2. use the retained summaries to recover the exact paper numbers without rerunning the full sweeps.

That is the same conceptual split used in the manuscript:

- the algorithm is easiest to understand on a small example,
- the final claims should be read from the retained large-scale datasets.
